In [ ]:
# 1. 외부 모듈 자동 새로고침 설정 (loader.py 수정 시 즉각 반영)
%load_ext autoreload
%autoreload 2

# 2. 필수 라이브러리 임포트
import os
import json
import pandas as pd
import FinanceDataReader as fdr
import pykrx
import OpenDartReader
import matplotlib
import seaborn
import scipy
from datetime import date

# 3. 직접 만든 로컬 모듈 임포트
from data.loader import QuantDataLoader

print("✅ 환경 설정 및 전체 라이브러리 정상 로드 완료!")

In [ ]:
import pickle

# 디버깅 분석을 위해 저장해둔 history 데이터 로드
try:
    with open('debug_history.pkl', 'rb') as f:
        history = pickle.load(f)
    print("✅ 'debug_history.pkl' 로드 완료! 이제 디버깅을 시작할 수 있습니다.")
except FileNotFoundError:
    print("❌ 'debug_history.pkl' 파일을 찾을 수 없습니다. 2번 노트북을 먼저 실행해주세요.")

print("==================================================")
print("🔍 파이프라인 탈락률(Funnel) 분석")
print("==================================================\n")

# 각 단계별 생존 종목 수 확인
stages = ['stage1', 'stage2', 'stage3', 'stage4', 'stage5']
previous_count = None

for stage in stages:
    if stage in history:
        current_count = len(history[stage])
        
        # 탈락률 계산
        if previous_count is not None and previous_count > 0:
            drop_rate = (previous_count - current_count) / previous_count * 100
            print(f"📉 {stage} 생존: {current_count}개 (직전 단계 대비 -{drop_rate:.1f}% 탈락)")
        else:
            print(f"🎯 {stage} 생존: {current_count}개 (섹터 통과 후 남은 전체 유니버스)")
            
        previous_count = current_count

# 만약 Stage 4까지 살아남았는데 Stage 5에서 다 죽었다면, 
# Stage 4의 생존자들을 살펴봅니다.
if 'stage4' in history and not history['stage4'].empty:
    print("\n💡 [마지막 생존자] Stage 4 통과 종목 (이들이 Stage 5에서 탈락함)")
    display(history['stage4'][['ticker', 'sector', 'roe', 'pbr']])

In [ ]:
import pandas as pd

print("==================================================")
print("🩺 [정밀 진단] Stage 3 탈락 종목 Raw Data 분석")
print("==================================================\n")

try:
    # 1. Stage 2 통과자 중 Stage 3에서 떨어진 종목 추출
    # history 딕셔너리에 데이터가 남아있다는 가정 하에 진행
    stage2_passed = history['stage2']
    stage3_passed_tickers = history['stage3']['ticker'].tolist() if not history['stage3'].empty else []
    
    stage3_failed_df = stage2_passed[~stage2_passed['ticker'].isin(stage3_passed_tickers)]
    sample_failures = stage3_failed_df.head(5)['ticker'].tolist()
    
    print(f"총 {len(stage3_failed_df)}개 종목이 Stage 3에서 탈락했습니다. 샘플 5개 진단 시작...\n")
    
    # 2. 로더를 직접 호출하여 데이터 계산 흐름 확인
    for ticker in sample_failures:
        print(f"▶️ 대상 종목: {ticker}")
        
        # 최근 6개 분기 시계열 로드
        q_series = loader.get_quarterly_financials_series(ticker, base_date, n_quarters=6)
        
        if len(q_series) < 6:
            print("  ❌ [사유] DATA_TOO_SHORT (가용 데이터 부족)\n")
            continue
            
        # 값 출력 (최근 3개 분기(t=0, 1, 2)와 전년 동기(t=4, 5, 6) 데이터 흐름 확인)
        for i, q in enumerate(q_series[:4]):  # 디버깅을 위해 최근 4분기만 출력
            rev = q.get('revenue', 0)
            sga = q.get('sga', 0)
            print(f"  [t-{i} 분기] 매출액: {rev:,.0f} | 판관비: {sga:,.0f}")
            
            if rev < 0 or sga < 0:
                print("  🚨 [경고] 음수 값 발견! 차분 로직(thstrm_add_amount) 오류 확실시 됨.")
                
        print("-" * 50)
        
except Exception as e:
    print(f"❌ 진단 중 에러 발생: {e}")

In [ ]:
import pandas as pd
from collections import Counter

print("==================================================")
print("🩺 Stage 3 진입 종목(75개) 전수 데이터 상태 판별 (로더 직접 호출)")
print("==================================================\n")

try:
    stage2_passed_df = history.get('stage2')
    if stage2_passed_df is None or stage2_passed_df.empty:
        print("Stage 2 통과 종목이 없습니다.")
    else:
        status_counter = Counter()
        
        print("⏳ 로더(loader)를 통해 75개 종목의 시계열 상태 직접 확인 중...")
        for _, row in stage2_passed_df.iterrows():
            ticker = row['ticker']
            
            # loader를 직접 호출하여 6분기 시계열 확보 시도 (에러 없이 가져오는지 확인)
            q_series = loader.get_quarterly_financials_series(ticker, base_date, n_quarters=6)
            
            if len(q_series) < 6:
                status_counter['DATA_TOO_SHORT (6분기 미달)'] += 1
            else:
                # 데이터가 6개 다 있다면, 음수나 결측치가 있는지 추가 확인
                has_error = False
                for q in q_series:
                    rev = q.get('revenue', 0)
                    sga = q.get('sga', 0)
                    # 데이터가 없거나(NaN), 음수값이 튀어나온 경우(단독값 파싱 에러 의심)
                    if pd.isna(rev) or pd.isna(sga) or rev < 0 or sga < 0:
                        has_error = True
                        break
                        
                if has_error:
                    status_counter['DATA_INVALID (음수 또는 결측치 포함)'] += 1
                else:
                    status_counter['COMPUTED (정상 계산 가능)'] += 1
            
        print("\n📊 [전수조사 결과: 데이터 상태 분포]")
        total = len(stage2_passed_df)
        for status, count in status_counter.items():
            print(f" - {status}: {count}개 종목 ({(count/total)*100:.1f}%)")

except Exception as e:
    print(f"❌ 진단 중 에러 발생: {e}")

In [ ]:
import pandas as pd

print("==================================================")
print("🕵️‍♂️ [심층 추적] 12개 비정상 데이터 원본 파싱 상태 점검")
print("==================================================\n")

try:
    stage2_passed_df = history['stage2']
    invalid_tickers = []
    
    for _, row in stage2_passed_df.iterrows():
        ticker = row['ticker']
        q_series = loader.get_quarterly_financials_series(ticker, base_date, n_quarters=6)
        
        # 6분기 데이터가 모두 존재하는 경우에만 값 검증
        if len(q_series) == 6:
            for q in q_series:
                rev = q.get('revenue', 0)
                sga = q.get('sga', 0)
                # 성장률(변동량)이 아닌, '매출액/판관비 절대치' 자체가 음수인 경우를 적발
                if pd.isna(rev) or pd.isna(sga) or rev < 0 or sga < 0:
                    invalid_tickers.append(ticker)
                    break
    
    print(f"발견된 의심 종목 수: {len(invalid_tickers)}개\n")
    
    # 비정상 데이터 상세 출력하여 원본 값 확인
    for ticker in invalid_tickers:
        print(f"▶️ 종목: {ticker}")
        q_series = loader.get_quarterly_financials_series(ticker, base_date, n_quarters=6)
        for i, q in enumerate(q_series):
            rev_val = q.get('revenue')
            sga_val = q.get('sga')
            
            rev_str = f"{rev_val:>20,}" if pd.notna(rev_val) else f"{'NaN':>20}"
            sga_str = f"{sga_val:>20,}" if pd.notna(sga_val) else f"{'NaN':>20}"
            
            print(f"  [t-{i}] 매출액: {rev_str} | 판관비: {sga_str}")
        print("-" * 50)
        
except Exception as e:
    print(f"❌ 에러: {e}")

In [ ]:
import os
import pandas as pd
from data.loader import QuantDataLoader

def scan_invalid_tickers_accounts():
    loader = QuantDataLoader(use_cache=True)
    
    # 🚨 여기에 DATA_INVALID 판정을 받은 13개 종목의 티커(문자열)를 입력하세요.
    invalid_tickers = ['103140', '069620', '003090', '032350', '185750',
                       '079160', '039130', '016590', '002310', '016800',
                       '037560', '032560', '019680'] # 예시 티커
    
    year = 2023
    reprt_code = '11011' # 사업보고서(11011) 등 결측치가 발생한 분기를 기준
    
    for ticker in invalid_tickers:
        print(f"\n{'='*60}")
        print(f"🔍 [{ticker}] {year}년 손익계산서(IS/CIS) 원본 계정명")
        print(f"{'='*60}")
        
        # 연결재무제표(CFS) 로드
        df = loader.get_financial_statements(ticker, year, reprt_code, fs_div='CFS')
        
        if df is None or df.empty:
            print("캐시된 데이터가 없거나 로드에 실패했습니다.")
            continue
            
        # 손익계산서 항목만 필터링
        is_df = df[df['sj_div'].isin(['IS', 'CIS'])]
        
        if is_df.empty:
            print("손익계산서(IS/CIS) 데이터가 존재하지 않습니다.")
            continue
            
        # 파악을 돕기 위해 계정명, 계정 ID, 당기 금액을 함께 출력
        display_cols = ['account_nm', 'account_id', 'thstrm_amount']
        
        # 판다스 출력 설정 (잘림 방지)
        pd.set_option('display.max_rows', None)
        output = is_df[display_cols].fillna('NaN')
        
        print(output.to_string(index=False))

if __name__ == "__main__":
    scan_invalid_tickers_accounts()

In [ ]:
# 파이프라인 실행 완료 후 (final_df, history = pipeline.run(...) 이후)

target_ticker = '103140'  # 테스트할 종목 코드 (판관비가 없어 NOT_COMPUTABLE이 예상되는 종목)

# 1. Stage 2 통과 여부 확인
stage2_df = history.get('stage2', pd.DataFrame())

if not stage2_df.empty and target_ticker in stage2_df['ticker'].values:
    print(f"✅ [{target_ticker}] Stage 2 생존 확인")
    
    # 2. Stage 3 통과 여부 및 데이터 확인
    stage3_df = history.get('stage3', pd.DataFrame())
    
    if not stage3_df.empty and target_ticker in stage3_df['ticker'].values:
        print(f"✅ [{target_ticker}] Stage 3 통과 (Exempt 구제 로직 정상 작동!)")
        
        # 실제 어떤 값으로 채워져서 통과했는지 해당 행 출력
        target_row = stage3_df[stage3_df['ticker'] == target_ticker]
        print("\n[Stage 3 데이터 확인]")
        print(target_row)
        
    else:
        print(f"🚨 [{target_ticker}] Stage 3에서 탈락(FAILED)했습니다.")
        print("결론: NOT_COMPUTABLE 구제 로직이 제대로 타지 않고 실격 처리 중입니다. Stage 3 코드 수정이 필요합니다.")
else:
    print(f"⚠️ [{target_ticker}] Stage 2에서 이미 탈락하여 Stage 3 검증이 불가능합니다. Stage 2를 통과한 다른 금융/지주사 종목으로 변경해주세요.")

In [ ]:
import yaml
import pandas as pd
from datetime import date
from IPython.display import display  # 주피터 환경용 깔끔한 출력

from backtest.forward_return import BacktestEngine
from core.pipeline import QuantPipeline 
from data.loader import QuantDataLoader

# 판다스 열(Column) 생략 방지
pd.set_option('display.max_columns', None)

def verify_backtest_engine_jupyter():
    # 1. 파라미터 로드 및 인스턴스 초기화
    with open("config/params.yaml", "r", encoding="utf-8") as f:
        params = yaml.safe_load(f)
        
    loader = QuantDataLoader(use_cache=True)
    pipeline = QuantPipeline(params, loader)
    engine = BacktestEngine(pipeline=pipeline, loader=loader)
    
    # 2. 테스트 구간 설정 (2개 분기)
    test_start = date(2023, 3, 1)
    test_end = date(2023, 9, 30)
    
    print("=== 백테스트 엔진 검증 시작 ===")
    perf_df, port_df = engine.run(test_start, test_end)
    
    # 3. 주피터 전용 출력 및 파일 저장
    print("\n=== [검증 결과 1] 포트폴리오 구성 로그 (샘플) ===")
    if not port_df.empty:
        display(port_df.head())  # 셀 출력 제한을 피하기 위해 일부만 표시
    else:
        print("포트폴리오 내역이 없습니다.")
    
    print("\n=== [검증 결과 2] 성과 요약 ===")
    if not perf_df.empty:
        display(perf_df)
    else:
        print("성과 내역이 없습니다.")
        
    # 전체 확인을 위한 CSV 추출
    port_df.to_csv("test_portfolio_log.csv", index=False, encoding="utf-8-sig")
    perf_df.to_csv("test_performance_log.csv", index=False, encoding="utf-8-sig")
    print("\n💡 전체 결과는 'test_portfolio_log.csv' 및 'test_performance_log.csv'로 저장되었습니다.")

# 셀 실행
verify_backtest_engine_jupyter()

In [ ]:
import logging
from datetime import date
import yaml
from core.pipeline import QuantPipeline
from data.loader import QuantDataLoader

def check_stage_by_stage_survival_safe():
    # 1. 파이프라인의 수많은 INFO 로그를 가려서 출력 창 폭발 방지
    logging.getLogger().setLevel(logging.WARNING)
    
    with open("config/params.yaml", "r", encoding="utf-8") as f:
        params = yaml.safe_load(f)
        
    # 캐시를 사용하므로 15분 대기 없이 즉각 실행됩니다.
    loader = QuantDataLoader(use_cache=True)
    pipeline = QuantPipeline(params, loader)
    
    test_dates = [date(2023, 3, 31), date(2023, 6, 30)]
    
    print("🚀 캐시 데이터를 읽어 빠르게 스크리닝을 진행 중입니다...")
    
    # 2. 콘솔 대신 텍스트 파일로 결과만 깔끔하게 저장
    with open("survival_report.txt", "w", encoding="utf-8") as out_f:
        out_f.write("=== 단계별 생존 종목 수 정밀 진단 ===\n")
        
        for bd in test_dates:
            out_f.write(f"\n[기준일: {bd}]\n")
            
            # 파이프라인 실행
            final_df, history = pipeline.run(bd)
            
            # 각 단계별 숫자 기록
            for stage_name, df in history.items():
                out_f.write(f" - {stage_name} 생존: {len(df)}개\n")
                
            out_f.write(f" => 🎯 최종 통과(final_df): {len(final_df)}개\n")
            
    print("✅ 진단 완료! 프로젝트 폴더의 'survival_report.txt' 파일을 열어 확인해 주세요.")

# 셀 실행
check_stage_by_stage_survival_safe()